# GreenSpace CNN - Core Pipeline v1

This is the PyTorch/TorchGeo handoff notebook for demonstrating the GreenSpace pipeline from Google Drive authentication through a trained-model evaluation. It calls reusable project functions instead of copying implementation logic from the historical notebooks.

## Current scaffold status

- Ready: project setup and explicit paths
- Ready: existing Google Drive OAuth authentication
- Pending packaging: rated-image download as a separate reusable command/function
- Ready: survey cleaning, inclusion filtering, and rater aggregation
- Ready: deterministic 50-image demonstration selection and 60/20/20 split
- Ready: current TorchGeo backbone/model configuration
- Ready: existing one-warm-up plus one-fine-tuning test orchestration
- Ready: existing evaluation and validation-threshold orchestration

> The 50-image, two-epoch run is functional smoke evidence only. Its metrics are not model-performance evidence and must not be compared with a full training run.


## 1. Setup and explicit inputs

Set `GREENSPACE_RAW_SURVEY` to the raw survey CSV before running the preprocessing cells. The expected headers and image layout are the same as the existing Notebook 02 workflow.


In [ ]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

raw_survey_env = os.getenv('GREENSPACE_RAW_SURVEY')
RAW_SURVEY_PATH = Path(raw_survey_env).expanduser() if raw_survey_env else None
IMAGE_CACHE_DIR = PROJECT_ROOT / 'data' / 'cache' / 'images'
DEMO_DATA_ROOT = PROJECT_ROOT / 'data' / 'core_pipeline_demo'
DEMO_SPLIT_DIR = DEMO_DATA_ROOT / 'processed' / 'splits'

PREPROCESSING_SEED = 123
SPLIT_SEED = 123
SPLIT_SEED_2 = 456
PYTORCH_SEED = 37
DEMO_SAMPLE_SIZE = 50

random.seed(PREPROCESSING_SEED)
np.random.seed(PREPROCESSING_SEED)

RUN_GOOGLE_AUTH = False
RUN_MODEL_BUILD = False
RUN_DEMO_TRAINING = False
RUN_DEMO_EVALUATION = False

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_SURVEY_PATH:', RAW_SURVEY_PATH or 'SET GREENSPACE_RAW_SURVEY')
print('IMAGE_CACHE_DIR:', IMAGE_CACHE_DIR)
print('DEMO_SPLIT_DIR:', DEMO_SPLIT_DIR)
print('Seeds:', {'preprocessing': PREPROCESSING_SEED, 'split_1': SPLIT_SEED, 'split_2': SPLIT_SEED_2, 'pytorch': PYTORCH_SEED})


## 2. Google Drive authentication

This uses the existing PyDrive2 OAuth flow in `src/drive_utils.py`. Google Cloud Console is used to create the OAuth desktop credentials; the images themselves are read from Google Drive. See `instruction_docs/google_drive_auth.md`.

Set `RUN_GOOGLE_AUTH = True` in the setup cell when you want the browser authentication flow to run.


In [ ]:
from src.drive_utils import get_drive

drive = None
if RUN_GOOGLE_AUTH:
    drive = get_drive(use_local_server=True)
    print('Google Drive authentication succeeded.')
else:
    print('Authentication skipped. Set RUN_GOOGLE_AUTH=True when Drive access is needed.')


## 3. Rated-image download - next packaging increment

The current rated-image download behavior is proven in Notebook 02, but its orchestration is still embedded in that notebook. The next implementation increment will extract it into a reusable Drive download function and a separate command such as:

```bash
python scripts/download_drive_images.py --manifest ... --cache-dir data/cache/images
```

This notebook will call the same Python function directly after that extraction. Until then, the following preprocessing and split cells expect the required rated images to already exist under `data/cache/images/`.


## 4. Clean and aggregate the survey

This section uses the functions extracted from the tested Notebook 02 behavior. It normalizes headers and filenames, keeps only `include_tile == yes`, and aggregates multiple rater rows into soft and hard labels.


In [ ]:
from src.preprocessing import clean_survey_dataframe, aggregate_rater_labels

if RAW_SURVEY_PATH is None:
    raise ValueError('Set GREENSPACE_RAW_SURVEY to the raw survey CSV before running this cell.')
if not RAW_SURVEY_PATH.is_file():
    raise FileNotFoundError(f'Missing raw survey CSV: {RAW_SURVEY_PATH}')

raw_survey = pd.read_csv(RAW_SURVEY_PATH)
cleaned_survey = clean_survey_dataframe(raw_survey, image_col='Image Name')
labels_soft, labels_hard, inclusion_summary = aggregate_rater_labels(cleaned_survey)

print('Inclusion summary:', inclusion_summary)
print('Soft-label images:', len(labels_soft))
print('Hard-label images:', len(labels_hard))
display(labels_soft.head())
display(labels_hard.head())


## 5. Select 50 images and create the demonstration split

The 50 images are selected deterministically with PyTorch seed `37`. The selected image-level labels are then split using the existing Notebook 02 seeds and ratios: 60% train, 20% validation, and 20% test. With 50 available cached images, this produces 30/10/10 manifests.


In [ ]:
from src.preprocessing import build_split_frames, write_split_frames

if len(labels_soft) < DEMO_SAMPLE_SIZE:
    raise ValueError(f'Demonstration requires at least {DEMO_SAMPLE_SIZE} unique included images; found {len(labels_soft)}.')

demo_names = labels_soft['image_filename'].sample(n=DEMO_SAMPLE_SIZE, random_state=PYTORCH_SEED).tolist()
demo_soft = labels_soft.set_index('image_filename').loc[demo_names].reset_index()
demo_hard = labels_hard.set_index('image_filename').loc[demo_names].reset_index()
missing_demo_images = [name for name in demo_names if not (IMAGE_CACHE_DIR / name).is_file()]
if missing_demo_images:
    raise FileNotFoundError(f'{len(missing_demo_images)} selected demo images are missing from {IMAGE_CACHE_DIR}. First example: {missing_demo_images[0]}')

demo_splits, demo_split_summary = build_split_frames(
    demo_soft,
    demo_hard,
    IMAGE_CACHE_DIR,
    split_seed=SPLIT_SEED,
    split_seed_2=SPLIT_SEED_2,
)
demo_split_paths = write_split_frames(demo_splits, DEMO_SPLIT_DIR)
print('Demo split summary:', demo_split_summary)
for split, path in demo_split_paths.items():
    print(f'{split}: {path}')


## 6. Inspect the current PyTorch/TorchGeo model

This uses the current canonical configuration: TorchGeo Swin V2 Base with NAIP RGB Satlas weights, seven active binary outputs, shade classification, and bounded continuous score/vegetation heads. Building the pretrained backbone may download its weights when they are not already cached.


In [ ]:
from src_torch.config import TORCH_DATA_CONFIG, TORCH_MODEL_CONFIG, TORCH_TRAINING_CONFIG
from src_torch.models import build_torchgeo_model
from src_torch.training import trainable_parameter_summary

print('Data configuration:', TORCH_DATA_CONFIG)
print('Model configuration:', TORCH_MODEL_CONFIG)
print('Training configuration:', TORCH_TRAINING_CONFIG)

demo_model = None
if RUN_MODEL_BUILD:
    demo_model = build_torchgeo_model()
    print('Model metadata:', demo_model.metadata())
    print('Parameter summary:', trainable_parameter_summary(demo_model))
else:
    print('Model build skipped. Set RUN_MODEL_BUILD=True to instantiate the configured backbone and heads.')


## 7. Run one warm-up epoch and one fine-tuning epoch

This calls the existing persistent PyTorch trainer with `test_run_mode=True`. The first epoch freezes the backbone and trains the GreenSpace heads. The second epoch unfreezes the configured backbone and fine-tunes end to end.

Set `RUN_DEMO_TRAINING = True` only after the 30/10/10 demo manifests exist.

Equivalent future terminal command:

```bash
python scripts/train_torch.py --data-root data/core_pipeline_demo --smoke --device auto
```


In [ ]:
training_result = None
if RUN_DEMO_TRAINING:
    if not all(path.is_file() for path in demo_split_paths.values()):
        raise FileNotFoundError('Create the demonstration split manifests before training.')
    os.environ['GREENSPACE_DATA_ROOT'] = str(DEMO_DATA_ROOT)
    from src_torch.training import run_persistent_warmup_finetune

    training_result = run_persistent_warmup_finetune(
        training_config={
            'seed': PYTORCH_SEED,
            'test_run_mode': True,
            'device': 'auto',
            'max_train_batches': None,
            'max_val_batches': None,
        }
    )
    display(training_result)
else:
    print('Demo training skipped. Set RUN_DEMO_TRAINING=True to run the existing 1+1 test schedule.')


## 8. Evaluate the trained demonstration model

This reuses the existing PyTorch evaluation functions: predict train/validation/test, calculate monitoring metrics, tune binary thresholds on validation, and save the evaluation artifacts beside the demo run.

Equivalent future terminal command:

```bash
python scripts/evaluate_torch.py --model-path models/runs/<demo-run>/best_mcmae_<demo-run>.pt --data-root data/core_pipeline_demo
```


In [ ]:
evaluation_paths = None
if RUN_DEMO_EVALUATION:
    if training_result is None:
        raise ValueError('Run the demonstration training cell before evaluation.')

    import torch
    from src_torch.data import load_split_dfs, resolve_split_schema
    from src_torch.evaluation import (
        evaluate_all_splits,
        evaluate_loss_monitoring,
        infer_run_tag_and_variant,
        load_torch_checkpoint_model,
        predict_split,
        save_evaluation_outputs,
        tune_thresholds_f1,
    )
    from src_torch.training import resolve_device

    checkpoint_path = Path(training_result['best_mc_mae_path'])
    device = resolve_device('auto')
    trained_model, _, _ = load_torch_checkpoint_model(checkpoint_path, device=device)
    run_tag, variant = infer_run_tag_and_variant(checkpoint_path)
    split_dfs = load_split_dfs()
    schema = resolve_split_schema(split_dfs['train'])

    predictions_by_split = {
        split: predict_split(trained_model, split, device=device, batch_size=int(TORCH_DATA_CONFIG['batch_size']))
        for split in ('train', 'val', 'test')
    }
    loss_monitor = evaluate_loss_monitoring(predictions_by_split, schema.binary_cols)
    val_df, val_predictions, _ = predictions_by_split['val']
    y_val = (val_df[schema.binary_cols].fillna(0.0).to_numpy(dtype=np.float32) >= 0.5).astype(int)
    thresholds = tune_thresholds_f1(y_val, val_predictions['bin_head'], schema.bin_names)
    threshold_map = {
        row['label']: float(row['best_threshold'])
        for _, row in thresholds.iterrows()
        if np.isfinite(row['best_threshold'])
    }
    overall, per_label = evaluate_all_splits(
        predictions_by_split,
        schema.binary_cols,
        threshold_map,
    )
    evaluation_paths = save_evaluation_outputs(
        run_tag=run_tag,
        variant=variant,
        loss_monitor_df=loss_monitor,
        thresholds_df=thresholds,
        overall_df=overall,
        per_label_df=per_label,
    )
    display(overall.round(4))
    display(per_label.round(4))
    print('Saved evaluation artifacts:', evaluation_paths)
else:
    print('Demo evaluation skipped. Set RUN_DEMO_EVALUATION=True after demo training succeeds.')


## 9. Production workflow after the demonstration

The completed handoff will expose separate commands for Drive download, preprocessing, training, evaluation, validation, and prediction. The notebook will continue to call the same underlying functions, while the terminal commands will support the full uncapped dataset derived from the approximately 10,000 raw survey rows.

The next implementation step is packaging the current rated-image Drive download behavior and wiring that function into Section 3.
